<a href="https://colab.research.google.com/github/Marcos-Med/IC/blob/main/FineTuningBERTimbau.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Realizando Fine Tuning no modelo BERTimbau
<p> Foi realizado um ajuste fino no modelo léxico BERTimbau, utilizando o corpus FakeBR. O objetivo do fine tuning é criar um modelo que preveja se uma notícia é verdadeira ou não.</p>

### Clonando o corpura que se encontra no repositório do GitHub

In [1]:
!git clone https://github.com/Marcos-Med/IC.git

Cloning into 'IC'...
remote: Enumerating objects: 21625, done.
remote: Counting objects: 100% (21625/21625), done.
remote: Compressing objects: 100% (14936/14936), done.
remote: Total 21625 (delta 6682), reused 21623 (delta 6682), pack-reused 0 (from 0)
Receiving objects: 100% (21625/21625), 21.53 MiB | 18.51 MiB/s, done.
Resolving deltas: 100% (6682/6682), done.
Updating files: 100% (21605/21605), done.


### Instalando as bibliotecas

In [2]:
!pip install transformers torch

In [66]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 14.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [72]:
# importando libs
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
import pandas as pd
import os
from datasets import Dataset
from sklearn.model_selection import KFold

### Importando os dados do Corpora FakeBR

In [52]:
def load_corpus_fakeBR():
    base_path = "/content/IC/corpura/fakebr/size_normalized_texts"

    fake_news = []
    true_news = []
    for root, dirs, files in os.walk(base_path + "/fake"):
        for file in files:
            if file.endswith('.txt'):
                with open(os.path.join(root, file), 'r') as f:
                    text = f.read()
                    fake_news.append([text, 0])

    for root, dirs, files in os.walk(base_path + "/true"):
        for file in files:
            if file.endswith('.txt'):
                with open(os.path.join(root, file), 'r') as f:
                    text = f.read()
                    true_news.append([text, 1])


    all_news = fake_news + true_news
    return pd.DataFrame(all_news, columns=['text', 'label'])

In [53]:
#Carregando o corpus fakeBR
fake_br = load_corpus_fakeBR()

In [57]:
fake_br.head()

,text,label
0,Palocci pode estar cogitando delação premiada....,0
1,Tem algo podre no ar!. O Diário do Brasil pub...,0
2,"Joesley xingando o Supremo: ""Nós precisamos or...",0
3,"Embaixador da Coreia do Norte em Londres: ""Não...",0
4,"Senadora Ana Amélia: ""Ele é do meu partido, m...",0


In [58]:
# Separar os dados textuais e a labels
text = fake_br['text']
labels = fake_br['label']

### Importando o modelo BERTimbau da NeuralMind

In [62]:
model_name = "neuralmind/bert-base-portuguese-cased"
tokenizer = BertTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

In [69]:
#Função para tokenizar os dados para o modelo BERTimbau
def tokenize(data):
  return tokenizer(data['text'], padding='max_length', truncation=True, max_length=512)


In [70]:
#Tokeniza os dados do Corpora FakeBR
dataset = Dataset.from_pandas(fake_br)
tokenized_dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/7200 [00:00<?, ? examples/s]

In [71]:
# Transformar o dataset em tensor do Pytorch
tokenized_dataset = tokenized_dataset.remove_columns(['text'])
tokenized_dataset.set_format("torch")

### Treinamento utilizando Validação Cruzada
**K** = 5 folds

In [73]:
# 5 folds para validação cruzada
Kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [74]:
results = []

In [77]:
os.environ["WANDB_DISABLED"] = "true"

In [79]:
for train_index, test_index, in Kf.split(tokenized_dataset):
    train_dataset = tokenized_dataset.select(train_index.tolist())
    test_dataset = tokenized_dataset.select(test_index.tolist())

    model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2) #Classificação binária

    # Paramêtros para realizar Fine Tuning no BERTimbau
    training_args = TrainingArguments(
        output_dir="./results",
        eval_strategy="epoch",
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        weight_decay=0.01,
        logging_dir="./logs",
        save_strategy="epoch"
        )
    #Objeto de treino
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset
    )

    #Treina o modelo
    trainer.train()

    #Avaliação do modelo BERTimbau
    metrics = trainer.evaluate()
    results.append(metrics)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Exception ignored in: <function _xla_gc_callback at 0x7efcbff749d0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/jax/_src/lib/__init__.py", line 96, in _xla_gc_callback
    def _xla_gc_callback(*args):
KeyboardInterrupt: 


Epoch,Training Loss,Validation Loss
1,0.146400,0.097263
2,0.036300,0.059758
3,0.003700,0.034159


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Epoch,Training Loss,Validation Loss
1,0.142400,0.037414
2,0.049000,0.024362


Epoch,Training Loss,Validation Loss
1,0.142400,0.037414
2,0.049000,0.024362
3,0.003900,0.024575


...

Epoch,Training Loss,Validation Loss
1,0.165900,0.023646
2,0.043900,0.033959
3,0.003100,0.028460


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Epoch,Training Loss,Validation Loss
1,0.145800,0.056107
2,0.054700,0.023880
3,0.003800,0.025258


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Epoch,Training Loss,Validation Loss
1,0.169500,0.067733
2,0.048100,0.115971
3,0.002700,0.051651
